<a href="https://colab.research.google.com/github/rockarus/csot-ml-astronomy/blob/main/submissions/arnesh/week1/week1_data_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import random
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
from torchvision.datasets import ImageFolder
import torchvision
import matplotlib.pyplot as plt

if torch.cuda.is_available():
  device = "cuda"
else:
  device = "cpu"
print("Device:", device)


Device: cpu


In [ ]:
from pathlib import Path

RAW_ROOT = Path("galaxy_raw")
IMAGES_DIR = RAW_ROOT / "images_gz2" / "images"
DATA_ROOT = Path("galaxy_data")
LABELS = "https://gz2hart.s3.amazonaws.com/gz2_hart16.csv.gz"

!pip -q install kaggle pandas
!kaggle datasets download -d jaimetrickz/galaxy-zoo-2-images
!unzip -q -o galaxy-zoo-2-images.zip -d {RAW_ROOT}
!wget -q -O {RAW_ROOT}/gz2_hart16.csv.gz {LABELS}
!gunzip -c {RAW_ROOT}/gz2_hart16.csv.gz > {RAW_ROOT}/gz2_hart16.csv
print("RAW_ROOT   =", RAW_ROOT)
print("IMAGES_DIR =", IMAGES_DIR)
print("DATA_ROOT  =", DATA_ROOT)

import os
print(os.listdir(IMAGES_DIR))
jpg_files = list(Path(IMAGES_DIR).rglob("*.jpg"))
print("JPG count:", len(jpg_files))
head = pd.read_csv(RAW_ROOT / "gz2_filename_mapping.csv", nrows=3)
print(head)

Dataset URL: https://www.kaggle.com/datasets/jaimetrickz/galaxy-zoo-2-images
License(s): Attribution 4.0 International (CC BY 4.0)
galaxy-zoo-2-images.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
def high_level_label(gz2_class):
  if gz2_class.startswith("E"):
    return "elliptical"
  elif gz2_class.startswith("SB"):
    return "spiral_barred"
  elif gz2_class.startswith("S"):
    return "spiral"
  else:
    return None

def build_imagefolder_layout(img_dir, map_csv, labels_csv, data_root, per_class, seed=42):
  mapping = pd.read_csv(map_csv)
  labels = pd.read_csv(labels_csv).rename(columns={"dr7objid": "objid"})
  df = mapping.merge(labels[["objid", "gz2_class"]], how = "inner", on = "objid")
  df["label"] = df["gz2_class"].map(high_level_label)
  df = df.dropna(subset=["label"])

  images_dir = Path(img_dir)
  out_root = Path(data_root)
  out_root.mkdir(parents=True, exist_ok=True)

  counts = {}
  for label in sorted(df["label"].unique()):
    class_dir = out_root / label
    class_dir.mkdir(exist_ok = True)
    rows = df[df["label"] == label]
    rows = rows.sample(n=min(len(rows), per_class), random_state = seed)
    link_ct = 0
    for i, row in rows.iterrows():
        src = images_dir / f"{int(row['asset_id'])}.jpg"
        dst = class_dir / f"{int(row['asset_id'])}.jpg"
        if src.exists() and not (dst.exists()):
            os.symlink(src.resolve(), dst)
            link_ct += 1
    counts[label] = link_ct
  return counts

pc=25000
counts = build_imagefolder_layout(IMAGES_DIR, RAW_ROOT / "gz2_filename_mapping.csv", RAW_ROOT / "gz2_hart16.csv", DATA_ROOT, pc)
print(counts)

In [ ]:
from torchvision.datasets import ImageFolder

img_str = ImageFolder(
    root=DATA_ROOT,
    transform=transforms.Compose([transforms.Resize((64, 64)), transforms.ToTensor()])
)
loader = DataLoader(img_str, batch_size=64, shuffle=False, num_workers=2)

n_pixels = 0
channel_sum = torch.zeros(3)
channel_sq_sum = torch.zeros(3)

for imgs, i in loader:
    channel_sum += imgs.sum(dim=[0, 2, 3])
    channel_sq_sum += (imgs ** 2).sum(dim=[0, 2, 3])
    n_pixels += imgs.shape[0] * imgs.shape[2] * imgs.shape[3]

mn = channel_sum / n_pixels
std_dev = (channel_sq_sum / n_pixels - mn ** 2).sqrt()
normal = transforms.Normalize(mean=mn.tolist(), std = std_dev.tolist())

dataset = ImageFolder(
    root=DATA_ROOT,
    transform=transforms.Compose([transforms.Resize((64, 64)), transforms.ToTensor(), normal])
)




In [ ]:
print("The number of images is", len(dataset))
print("The classes are", dataset.classes)
print("class_to_idx:", dataset.class_to_idx)
image, label = dataset[0]
print("The image shape is", image.shape)   # torch.Size([3, 64, 64])
print("The image datatype is", image.dtype)   # torch.float32
print("Label:", dataset.classes[label])

In [ ]:
loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)

images, labels = next(iter(loader))
print("Image shape:", images.shape)
print("Label shape:", labels.shape)

In [ ]:
images_show = images * 0.5 + 0.5
grid = torchvision.utils.make_grid(images_show[:8], nrow=8)

plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
plt.axis("off")
plt.title("Good plot")
plt.show()

print("Labels:", [dataset.classes[i] for i in labels[:8].tolist()])